# Scraping de YouTube para moderacion de contenido

**Politica y farandula peruana**

**Objetivo:** recolectar metadatos y transcripciones publicas sin usar APIs ni llaves externas. El cuaderno sigue el estilo de los ejemplos de clase: instalacion, exploracion, scraping, organizacion en tablas y almacenamiento local.

**Flujo:**
1. Definir canales semilla.
2. Buscar canales candidatos con Selenium sobre la web publica.
3. Extraer videos recientes con `yt-dlp` sin descargar video.
4. Descargar subtitulos/transcripciones publicas con `yt-dlp`.
5. Guardar un archivo `jsonl` compatible con el cuaderno de limpieza.

## 1. Instalacion de librerias

Estas librerias se usan de manera local. No se usa YouTube Data API, Google API ni servicios de LLM.

In [17]:
# Instalación de librerías del proyecto
# youtube-transcript-api es un scraper sin API key (fallback para transcripciones)
!pip install -q pandas selenium beautifulsoup4 webdriver-manager yt-dlp tqdm youtube-transcript-api



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import ssl
import time
import urllib3
from urllib.parse import quote_plus

# ── Parche SSL para proxy corporativo con certificado autofirmado ─────────────
# El proxy intercepta TLS y presenta su propio cert; se deshabilita la
# verificación estricta sólo para esta sesión del notebook.
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'datos' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Directorio de trabajo:', ROOT)
print('Salida raw:', RAW_DIR)
print('SSL check: deshabilitado (proxy corporativo)')


Directorio de trabajo: C:\usr\ths_mia_fiis\pln\trabajo
Salida raw: C:\usr\ths_mia_fiis\pln\trabajo\datos\raw


## 2. Canales semilla

La lista inicial no es un ranking cerrado. Sirve para iniciar el corpus y debe revisarse manualmente segun disponibilidad de transcripciones, calidad de audio y pertinencia del contenido.

In [19]:
canales_semilla = pd.DataFrame([
    {'nombre': 'Marco Sifuentes / Ocram', 'categoria': 'politica_analisis', 'url': 'https://www.youtube.com/user/ocram001', 'nota': 'La Encerrona y entrevistas; lenguaje politico explicativo'},
    {'nombre': 'El diario de Curwen', 'categoria': 'politica_opinion', 'url': 'https://www.youtube.com/@curwen', 'nota': 'opinion politica, streaming y satira'},
    {'nombre': 'Sin Guion con Rosa Maria Palacios', 'categoria': 'politica_periodismo', 'url': 'https://www.youtube.com/@singuionlr', 'nota': 'periodismo de opinion y entrevistas'},
    {'nombre': 'RPP Noticias', 'categoria': 'politica_actualidad', 'url': 'https://www.youtube.com/@RPPNoticias', 'nota': 'noticias y entrevistas con lenguaje formal'},
    {'nombre': 'Exitosa Noticias', 'categoria': 'politica_actualidad', 'url': 'https://www.youtube.com/@exitosanoticias', 'nota': 'noticias, entrevistas y opinion'},
    {'nombre': 'Willax Television', 'categoria': 'politica_opinion', 'url': 'https://www.youtube.com/@WillaxTV', 'nota': 'programas de opinion politica'},
    {'nombre': 'Canal N', 'categoria': 'politica_actualidad', 'url': 'https://www.youtube.com/@canaln', 'nota': 'noticias y entrevistas'},
    {'nombre': 'Todo Good', 'categoria': 'streaming_humor_opinion', 'url': 'https://www.youtube.com/@todogoodpe', 'nota': 'conversacion, humor, invitados y coyuntura'},
    {'nombre': 'Hablando Huevadas', 'categoria': 'humor_streaming', 'url': 'https://www.youtube.com/@HablandoHuevadasOficial', 'nota': 'humor adulto; util para lenguaje ofensivo y contexto sensible'},
    {'nombre': 'Magaly TV La Firme', 'categoria': 'farandula', 'url': 'https://www.youtube.com/@MagalyTVLaFirme', 'nota': 'farandula, conflicto publico y lenguaje de espectaculos'},
    {'nombre': 'Amor y Fuego', 'categoria': 'farandula', 'url': 'https://www.youtube.com/@AmoryFuego', 'nota': 'farandula y comentarios de entretenimiento'},
    {'nombre': 'America Hoy', 'categoria': 'farandula', 'url': 'https://www.youtube.com/@americahoytv', 'nota': 'magazine y entretenimiento; verificar disponibilidad de videos'},
    {'nombre': 'Misias pero viajeras', 'categoria': 'viajes', 'url': 'https://www.youtube.com/c/Misiasperoviajeras', 'nota': 'archivo de viajes; canal cerrado pero episodios utiles'},
    {'nombre': 'Buen Viaje', 'categoria': 'viajes', 'url': 'https://www.youtube.com/c/BuenViajePe', 'nota': 'viajes por Peru con lenguaje descriptivo'},
    {'nombre': 'Viaja y Prueba', 'categoria': 'viajes_gastronomia', 'url': 'https://www.youtube.com/@ViajayPrueba', 'nota': 'viajes y gastronomia por Peru'},
])

candidatos_por_verificar = pd.DataFrame([
    {'nombre': 'Cacas / El Cacas', 'categoria': 'humor_streaming', 'consulta': 'Cacas youtuber peruano canal oficial YouTube', 'nota': 'pendiente de confirmar canal oficial antes de descargar'},
    {'nombre': 'Goblinciano', 'categoria': 'streaming_opinion', 'consulta': 'Goblinciano YouTube Peru canal oficial', 'nota': 'verificar pertinencia y estabilidad del contenido'},
    {'nombre': 'Instarandula', 'categoria': 'farandula_digital', 'consulta': 'Instarandula Samuel Suarez YouTube canal oficial', 'nota': 'verificar canal oficial y formato de episodios'},
])

canales_semilla.to_csv(RAW_DIR / 'canales_semilla.csv', index=False)
candidatos_por_verificar.to_csv(RAW_DIR / 'candidatos_por_verificar.csv', index=False)
canales_semilla

,nombre,categoria,url,nota
0,Marco Sifuentes / Ocram,politica_analisis,https://www.youtube.com/user/ocram001,La Encerrona y entrevistas; lenguaje politico ...
1,El diario de Curwen,politica_opinion,https://www.youtube.com/@curwen,"opinion politica, streaming y satira"
2,Sin Guion con Rosa Maria Palacios,politica_periodismo,https://www.youtube.com/@singuionlr,periodismo de opinion y entrevistas
3,RPP Noticias,politica_actualidad,https://www.youtube.com/@RPPNoticias,noticias y entrevistas con lenguaje formal
4,Exitosa Noticias,politica_actualidad,https://www.youtube.com/@exitosanoticias,"noticias, entrevistas y opinion"
5,Willax Television,politica_opinion,https://www.youtube.com/@WillaxTV,programas de opinion politica
6,Canal N,politica_actualidad,https://www.youtube.com/@canaln,noticias y entrevistas
7,Todo Good,streaming_humor_opinion,https://www.youtube.com/@todogoodpe,"conversacion, humor, invitados y coyuntura"
8,Hablando Huevadas,humor_streaming,https://www.youtube.com/@HablandoHuevadasOficial,humor adulto; util para lenguaje ofensivo y co...
9,Magaly TV La Firme,farandula,https://www.youtube.com/@MagalyTVLaFirme,"farandula, conflicto publico y lenguaje de esp..."


## 3. Busqueda web con Selenium

Esta seccion reproduce la logica de scraping vista en clase: abrir navegador, cargar una pagina, obtener HTML y parsear enlaces. Si ya se tienen URLs verificadas, esta parte puede omitirse.

In [20]:
UA = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/124.0.0.0 Safari/537.36'
)


def iniciar_driver(headless=True):
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1400,1000')
    options.add_argument('--lang=es-PE')
    options.add_argument(f'--user-agent={UA}')
    # Ocultar señales de automatización
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    # Eliminar propiedad webdriver del navegador
    driver.execute_cdp_cmd(
        'Page.addScriptToEvaluateOnNewDocument',
        {'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'}
    )
    return driver


def buscar_canales_youtube(consulta, n_scrolls=2, headless=True):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    driver = iniciar_driver(headless=headless)
    try:
        # &sp=EgIQAg%3D%3D filtra resultados por tipo 'Canal'
        url = ('https://www.youtube.com/results?search_query='
               + quote_plus(consulta) + '&sp=EgIQAg%3D%3D')
        driver.get(url)
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, 'ytd-channel-renderer'))
            )
        except Exception:
            time.sleep(4)  # fallback si ytd-channel-renderer no aparece
        for _ in range(n_scrolls):
            driver.execute_script('window.scrollTo(0, document.documentElement.scrollHeight);')
            time.sleep(2)
        html = driver.page_source
    finally:
        driver.quit()

    soup = BeautifulSoup(html, 'html.parser')
    filas = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        texto = a.get_text(' ', strip=True)
        if ('/@' in href or '/channel/' in href or '/c/' in href) and texto:
            if href.startswith('/'):
                href = 'https://www.youtube.com' + href
            filas.append({'consulta': consulta, 'nombre': texto, 'url': href.split('?')[0]})
    return pd.DataFrame(filas).drop_duplicates('url')


consultas = [
    'politica peru analisis canal youtube',
    'noticias politica peru youtube',
    'Marco Sifuentes Ocram YouTube Peru',
    'Curwen YouTube Peru canal oficial',
    'Rosa Maria Palacios Sin Guion YouTube Peru',
    'Hablando Huevadas YouTube Peru canal oficial',
    'Cacas youtuber peruano canal oficial YouTube',
    'farandula peru programa youtube',
    'Magaly Amor y Fuego America Hoy YouTube Peru',
    'youtubers peruanos viajes Misias Buen Viaje Viaja y Prueba',
]


In [21]:
# Ejecutar esta celda si se desea descubrir nuevos canales desde la web publica.
# Puede tardar por la carga dinamica de YouTube.

ejecutar_busqueda = False

if ejecutar_busqueda:
    candidatos = []
    for consulta in tqdm(consultas):
        candidatos.append(buscar_canales_youtube(consulta, n_scrolls=2, headless=True))
    canales_candidatos = pd.concat(candidatos, ignore_index=True).drop_duplicates('url')
    canales_candidatos.to_csv(RAW_DIR / 'canales_candidatos_scraping.csv', index=False)
else:
    canales_candidatos = canales_semilla.copy()

canales_candidatos.head(20)

,nombre,categoria,url,nota
0,Marco Sifuentes / Ocram,politica_analisis,https://www.youtube.com/user/ocram001,La Encerrona y entrevistas; lenguaje politico ...
1,El diario de Curwen,politica_opinion,https://www.youtube.com/@curwen,"opinion politica, streaming y satira"
2,Sin Guion con Rosa Maria Palacios,politica_periodismo,https://www.youtube.com/@singuionlr,periodismo de opinion y entrevistas
3,RPP Noticias,politica_actualidad,https://www.youtube.com/@RPPNoticias,noticias y entrevistas con lenguaje formal
4,Exitosa Noticias,politica_actualidad,https://www.youtube.com/@exitosanoticias,"noticias, entrevistas y opinion"
5,Willax Television,politica_opinion,https://www.youtube.com/@WillaxTV,programas de opinion politica
6,Canal N,politica_actualidad,https://www.youtube.com/@canaln,noticias y entrevistas
7,Todo Good,streaming_humor_opinion,https://www.youtube.com/@todogoodpe,"conversacion, humor, invitados y coyuntura"
8,Hablando Huevadas,humor_streaming,https://www.youtube.com/@HablandoHuevadasOficial,humor adulto; util para lenguaje ofensivo y co...
9,Magaly TV La Firme,farandula,https://www.youtube.com/@MagalyTVLaFirme,"farandula, conflicto publico y lenguaje de esp..."


## 4. Extraccion de videos con yt-dlp

`yt-dlp` permite leer metadatos publicos y subtitulos sin usar una API key. Para mantener el corpus controlado, se limita el numero de videos por canal.

In [ ]:
import yt_dlp

YT_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    )
}

# Opciones base compartidas entre listar y descargar
_YT_BASE_OPTS = {
    'quiet': True,
    'no_warnings': True,
    'ignoreerrors': True,
    'nocheckcertificate': True,   # fix: proxy corporativo con cert autofirmado
    'extractor_retries': 3,
    'http_headers': YT_HEADERS,
    'sleep_interval': 1,
    'max_sleep_interval': 3,
}


def _normalizar_url_canal(channel_url):
    """Garantiza que la URL apunte a la pestaña /videos del canal."""
    base = channel_url.rstrip('/')
    for sufijo in ('/videos', '/streams', '/shorts', '/playlists'):
        if base.endswith(sufijo):
            base = base[: -len(sufijo)]
    return base + '/videos'


def listar_videos_canal(channel_url, max_videos=15):
    url_videos = _normalizar_url_canal(channel_url)
    opciones = {
        **_YT_BASE_OPTS,
        'extract_flat': 'in_playlist',
        'playlist_items': f'1:{max_videos}',
        'skip_download': True,
    }
    try:
        with yt_dlp.YoutubeDL(opciones) as ydl:
            info = ydl.extract_info(url_videos, download=False)
    except Exception as exc:
        print(f'  ✗ Error extrayendo canal {channel_url}: {exc}')
        return []

    if not info:
        return []

    filas = []
    for item in info.get('entries', []) or []:
        if not item:
            continue
        video_id = item.get('id')
        if not video_id:
            continue
        filas.append({
            'video_id': video_id,
            'url': f'https://www.youtube.com/watch?v={video_id}',
            'title': item.get('title'),
            'upload_date': item.get('upload_date'),
            'duration': item.get('duration'),
            'view_count': item.get('view_count'),
            'channel_url': channel_url,
        })
    return filas


In [25]:
MAX_VIDEOS_POR_CANAL = 20

videos = []
for _, canal in tqdm(canales_semilla.iterrows(), total=len(canales_semilla)):
    try:
        filas = listar_videos_canal(canal['url'], max_videos=MAX_VIDEOS_POR_CANAL)
        for fila in filas:
            fila['channel_title'] = canal['nombre']
            fila['categoria_fuente'] = canal['categoria']
        videos.extend(filas)
    except Exception as exc:
        print('No se pudo leer canal:', canal['nombre'], exc)

videos_df = pd.DataFrame(videos).drop_duplicates('video_id')
videos_df.to_csv(RAW_DIR / 'videos_candidatos.csv', index=False)
videos_df.head()

  0%|          | 0/15 [00:00<?, ?it/s]

ERROR: [youtube:tab] ocram001/videos: Unable to download API page: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010) (caused by CertificateVerifyError('[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010)')); please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
ERROR: [youtube:tab] @curwen/videos: Unable to download API page: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010) (caused by CertificateVerifyError('[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010)')); please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the

""


## 5. Descarga de subtitulos publicos

Se descargan subtitulos en espanol cuando existan. Si un video no tiene subtitulos, queda fuera del corpus inicial o se transcribe luego con un modelo ASR local.

In [ ]:
SUBS_DIR = RAW_DIR / 'subtitulos'
SUBS_DIR.mkdir(parents=True, exist_ok=True)


def descargar_subtitulos(video_url, video_id):
    """Descarga subtítulos en español (manuales o auto-generados).
    Devuelve el Path al .vtt o None si no hay subtítulos disponibles.
    """
    outtmpl = str(SUBS_DIR / f'{video_id}.%(ext)s')
    opciones = {
        **_YT_BASE_OPTS,
        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': ['es', 'es-419', 'es-PE'],
        'subtitlesformat': 'vtt',
        'outtmpl': outtmpl,
    }
    try:
        with yt_dlp.YoutubeDL(opciones) as ydl:
            ydl.download([video_url])
    except Exception:
        pass
    archivos = sorted(SUBS_DIR.glob(f'{video_id}*.vtt'))
    return archivos[0] if archivos else None


def descargar_subtitulos_transcript_api(video_id):
    """Fallback: obtiene transcripción vía youtube-transcript-api.
    El parche ssl del inicio cubre también las requests de esta librería.
    """
    try:
        from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        try:
            t = transcript_list.find_manually_created_transcript(['es', 'es-419', 'es-PE'])
        except NoTranscriptFound:
            t = transcript_list.find_generated_transcript(['es', 'es-419', 'es-PE'])
        return [{'start': s['start'], 'duration': s['duration'], 'text': s['text']}
                for s in t.fetch()]
    except Exception:
        return []


def tiempo_a_segundos(valor):
    partes = valor.replace(',', '.').split(':')
    partes = [float(p) for p in partes]
    if len(partes) == 3:
        h, m, s = partes
    else:
        h, m, s = 0, partes[0], partes[1]
    return h * 3600 + m * 60 + s


def limpiar_linea_vtt(linea):
    linea = re.sub(r'<[^>]+>', '', linea)
    linea = re.sub(r'&amp;', '&', linea)
    linea = re.sub(r'&nbsp;', ' ', linea)
    linea = re.sub(r'\s+', ' ', linea).strip()
    return linea


def leer_vtt(path):
    texto = path.read_text(encoding='utf-8', errors='ignore')
    bloques = re.split(r'\n\s*\n', texto)
    segmentos = []
    vistos = set()
    patron_tiempo = re.compile(
        r'(\d{2}:\d{2}:\d{2}\.\d{3}|\d{2}:\d{2}\.\d{3})'
        r'\s+-->\s+'
        r'(\d{2}:\d{2}:\d{2}\.\d{3}|\d{2}:\d{2}\.\d{3})'
    )
    for bloque in bloques:
        lineas = [ln.strip() for ln in bloque.splitlines() if ln.strip()]
        if not lineas:
            continue
        match = None
        idx = 0
        for i, linea in enumerate(lineas):
            match = patron_tiempo.search(linea)
            if match:
                idx = i
                break
        if not match:
            continue
        start = tiempo_a_segundos(match.group(1))
        end = tiempo_a_segundos(match.group(2))
        frase = ' '.join(limpiar_linea_vtt(ln) for ln in lineas[idx + 1:])
        frase = re.sub(r'\s+', ' ', frase).strip()
        clave = (round(start, 1), frase.lower())
        if frase and clave not in vistos:
            vistos.add(clave)
            segmentos.append({'start': start, 'duration': max(end - start, 0.1), 'text': frase})
    return segmentos


In [ ]:
def hash_texto(texto):
    return hashlib.md5(texto.encode('utf-8')).hexdigest()


transcripciones = []
sin_subs = []

for _, video in tqdm(videos_df.iterrows(), total=len(videos_df)):
    vid = video['video_id']
    segmentos = []
    fuente_subs = None

    # --- Intento 1: yt-dlp (subtítulos del archivo .vtt) ---
    try:
        vtt_path = descargar_subtitulos(video['url'], vid)
        if vtt_path:
            segmentos = leer_vtt(vtt_path)
            fuente_subs = 'yt-dlp-vtt'
    except Exception as exc:
        print(f'  yt-dlp falló en {vid}: {exc}')

    # --- Intento 2: youtube-transcript-api como fallback ---
    if not segmentos:
        segmentos = descargar_subtitulos_transcript_api(vid)
        if segmentos:
            fuente_subs = 'transcript-api'

    texto = ' '.join(seg['text'] for seg in segmentos)
    if len(texto) < 200:
        sin_subs.append(vid)
        continue

    transcripciones.append({
        'video_id': vid,
        'url': video['url'],
        'title': video.get('title'),
        'channel_title': video.get('channel_title'),
        'categoria_fuente': video.get('categoria_fuente'),
        'fuente_subs': fuente_subs,
        'text_hash': hash_texto(texto),
        'segments': segmentos,
    })

salida = RAW_DIR / 'transcripts_raw.jsonl'
with open(salida, 'w', encoding='utf-8') as f:
    for row in transcripciones:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'Transcripciones guardadas : {len(transcripciones)}')
print(f'Sin subtítulos (omitidos) : {len(sin_subs)}')
print(f'Archivo: {salida}')


## 6. Revision rapida

Antes de pasar al cuaderno 02, se recomienda revisar manualmente canales, videos y transcripciones para retirar contenido con audio deficiente, publicidad extensa o segmentos que no pertenecen al objetivo del corpus.

In [ ]:
resumen = pd.DataFrame([
    {'archivo': 'canales_semilla.csv', 'ruta': str(RAW_DIR / 'canales_semilla.csv')},
    {'archivo': 'videos_candidatos.csv', 'ruta': str(RAW_DIR / 'videos_candidatos.csv')},
    {'archivo': 'transcripts_raw.jsonl', 'ruta': str(RAW_DIR / 'transcripts_raw.jsonl')},
])
resumen